# Solved-State Covariance Verification

This notebook verifies the new solved-state covariance support in `adam_core`.

It focuses on three questions:

- Can we preserve full `6 + k` solved-state covariance through coordinate conversions?
- Can we recover the orbital `6x6` block from a `7x7`, `8x8`, or `9x9` solved-state covariance and show that it matches the ordinary coordinate covariance?
- Can we use the solved-state covariance for variant creation and grouped collapse while keeping non-gravitational parameters aligned with the sampled orbital state?

The examples below use both synthetic cases and real SBDB / NEOCC fixtures already checked into the repository.


## What This Notebook Covers

We will walk through:

1. A synthetic `9x9` Keplerian example with `A1/A2/A3`.
2. A synthetic `7x7` grouped-collapse example with `A2`.
3. A real SBDB `99942` example with an `8x8` solved-state covariance (`A1/A2`).
4. A real NEOCC `99942` example with a `7x7` solved-state covariance (`A2`).

The notebook is deliberately centered on covariance integrity rather than propagation accuracy. The earlier non-grav support notebook is still the better place to inspect the propagation behavior itself.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from adam_core.coordinates import (
    CartesianCoordinates,
    CoordinateCovariances,
    KeplerianCoordinates,
    Origin,
    transform_coordinates,
)
from adam_core.coordinates.covariances import transform_solved_state_covariances_jacobian
from adam_core.orbits import Orbits
from adam_core.orbits.non_gravitational_parameters import NonGravitationalParameters
from adam_core.orbits.physical_parameters import PhysicalParameters
from adam_core.orbits.query.neocc import (
    _non_gravitational_parameters_from_neocc,
    _parse_oef,
    _solve_for_codes_to_names,
)
from adam_core.orbits.query.sbdb import _orbits_from_sbdb_payloads
from adam_core.orbits.solved_state_covariances import (
    ORBITAL_PARAMETER_NAMES,
    SolvedStateCovariances,
)
from adam_core.orbits.variants import VariantOrbits
from adam_core.time import Timestamp

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
np.set_printoptions(precision=6, suppress=False)


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "adam_core").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing src/adam_core")


ROOT = find_repo_root(Path.cwd().resolve())
TESTDATA = ROOT / "src" / "adam_core" / "orbits" / "query" / "tests" / "testdata"
SBDB_DIR = TESTDATA / "sbdb"
NEOCC_DIR = TESTDATA / "neocc"


In [ ]:
def orbital_covariance_frame(label: str, coords, solved: SolvedStateCovariances) -> pd.DataFrame:
    orbital = solved.to_orbital_covariances().to_matrix()[0]
    coordinate = coords.covariance.to_matrix()[0]
    return pd.DataFrame(
        {
            "metric": [
                "coordinate covariance shape",
                "solved-state covariance dimension",
                "max |orbital block - coordinate covariance|",
            ],
            "value": [
                str(coordinate.shape),
                solved.dimension[0].as_py(),
                float(np.max(np.abs(orbital - coordinate))),
            ],
        }
    ).assign(case=label)


def compare_orbital_blocks(label: str, left: np.ndarray, right: np.ndarray) -> pd.DataFrame:
    diff = left - right
    return pd.DataFrame(
        {
            "case": [label],
            "max_abs_difference": [float(np.max(np.abs(diff)))],
            "frobenius_norm": [float(np.linalg.norm(diff))],
        }
    )


def load_sbdb_orbit(fixture_name: str, object_id: str | None = None) -> Orbits:
    payload = json.loads((SBDB_DIR / fixture_name).read_text())
    if object_id is None:
        object_id = Path(fixture_name).stem.replace("_phys", "")
    return _orbits_from_sbdb_payloads([object_id], [payload])


def load_neocc_orbit(fixture_name: str) -> Orbits:
    data = _parse_oef((NEOCC_DIR / fixture_name).read_text())
    time_scale = data["time_system"].lower()
    if time_scale == "tdt":
        time_scale = "tt"

    solve_for = _solve_for_codes_to_names(
        (data.get("nongrav") or {}).get("solve_for_parameter_codes") or []
    )

    keplerian = KeplerianCoordinates.from_kwargs(
        a=[data["elements"]["a"]],
        e=[data["elements"]["e"]],
        i=[data["elements"]["i"]],
        raan=[data["elements"]["node"]],
        ap=[data["elements"]["peri"]],
        M=[data["elements"]["M"]],
        time=Timestamp.from_mjd([data["epoch"]], scale=time_scale),
        covariance=CoordinateCovariances.from_matrix(data["covariance"].reshape(1, 6, 6)),
        frame="ecliptic",
        origin=Origin.from_kwargs(code=["SUN"]),
    )

    from adam_core.coordinates.transform import _keplerian_to_cartesian_a

    solved_cartesian = transform_solved_state_covariances_jacobian(
        keplerian.values,
        [data.get("covariance_full")],
        _keplerian_to_cartesian_a,
        in_axes=(0, 0, None, None),
        out_axes=0,
        mu=keplerian.origin.mu(),
        max_iter=1000,
        tol=1e-15,
    )

    return Orbits.from_kwargs(
        orbit_id=[data["object_id"]],
        object_id=[data["object_id"]],
        coordinates=keplerian.to_cartesian(),
        physical_parameters=PhysicalParameters.from_kwargs(
            H_v=[data.get("magnitude", {}).get("H")],
            H_v_sigma=[np.nan],
            G=[data.get("magnitude", {}).get("G")],
            G_sigma=[np.nan],
        ),
        non_gravitational_parameters=_non_gravitational_parameters_from_neocc(data),
        solved_state_covariance=SolvedStateCovariances.from_matrix(
            solved_cartesian,
            [list(ORBITAL_PARAMETER_NAMES) + solve_for] if data.get("covariance_full") is not None else [None],
        ),
    )


def nongrav_summary(orbits: Orbits) -> pd.DataFrame:
    ng = orbits.non_gravitational_parameters
    solved = orbits.solved_state_covariance
    return pd.DataFrame(
        {
            "object_id": orbits.object_id.to_pylist(),
            "model": ng.model.to_pylist(),
            "solution_dimension": ng.solution_dimension.to_pylist(),
            "estimated_parameter_names": ng.estimated_parameter_names.to_pylist(),
            "solved_state_dimension": solved.dimension.to_pylist(),
            "solved_state_parameter_names": solved.parameter_names.to_pylist(),
            "A1": ng.A1.to_pylist(),
            "A2": ng.A2.to_pylist(),
            "A3": ng.A3.to_pylist(),
            "DT": ng.DT.to_pylist(),
            "AMRAT": ng.AMRAT.to_pylist(),
            "RHO": ng.RHO.to_pylist(),
        }
    )


def assert_close(label: str, left: np.ndarray, right: np.ndarray, atol: float = 1e-10) -> None:
    np.testing.assert_allclose(left, right, rtol=0, atol=atol)
    display(Markdown(f"**{label}:** passed with `atol={atol}`"))


## Synthetic Case 1: `9x9` Keplerian Solved-State Covariance

This is the cleanest demonstration of the full `6 + 3` path.

We create a Keplerian orbit with a `9x9` covariance whose solved parameters are:

- the six orbital elements `a, e, i, raan, ap, M`
- `A1, A2, A3`

Then we convert it to Cartesian and back, and verify that:

- the full `9x9` matrix survives the round-trip
- the recovered orbital `6x6` matches the original coordinate covariance
- the round-tripped coordinate covariance and the recovered `6x6` still agree


In [ ]:
keplerian_covariance = np.diag(
    [1e-10, 2e-10, 3e-8, 4e-8, 5e-8, 6e-8, 9e-26, 4e-26, 1e-26]
).reshape(1, 9, 9)
keplerian_covariance[0, 1, 6] = 2e-18
keplerian_covariance[0, 6, 1] = 2e-18
keplerian_covariance[0, 5, 7] = -1e-18
keplerian_covariance[0, 7, 5] = -1e-18
keplerian_covariance[0, 3, 8] = 5e-19
keplerian_covariance[0, 8, 3] = 5e-19

keplerian = KeplerianCoordinates.from_kwargs(
    a=[1.2],
    e=[0.15],
    i=[7.0],
    raan=[30.0],
    ap=[45.0],
    M=[12.0],
    covariance=CoordinateCovariances.from_matrix(keplerian_covariance[:, :6, :6]),
    time=Timestamp.from_mjd([60000.0], scale="tdb"),
    origin=Origin.from_kwargs(code=["SUN"]),
    frame="ecliptic",
)

solved_9x9 = SolvedStateCovariances.from_matrix(
    keplerian_covariance,
    [["a", "e", "i", "raan", "ap", "M", "A1", "A2", "A3"]],
)

cartesian_from_keplerian, solved_cartesian_9x9 = transform_coordinates(
    keplerian,
    representation_out=CartesianCoordinates,
    solved_state_covariances=solved_9x9,
)
keplerian_roundtrip, solved_roundtrip_9x9 = transform_coordinates(
    cartesian_from_keplerian,
    representation_out=KeplerianCoordinates,
    solved_state_covariances=solved_cartesian_9x9,
)

display(pd.concat([
    orbital_covariance_frame("original keplerian", keplerian, solved_9x9),
    orbital_covariance_frame("roundtrip keplerian", keplerian_roundtrip, solved_roundtrip_9x9),
], ignore_index=True))

display(compare_orbital_blocks(
    "original 6x6 vs recovered roundtrip 6x6",
    keplerian.covariance.to_matrix()[0],
    solved_roundtrip_9x9.to_orbital_covariances().to_matrix()[0],
))

assert_close(
    "Recovered 6x6 matches original Keplerian covariance",
    solved_roundtrip_9x9.to_orbital_covariances().to_matrix(),
    keplerian.covariance.to_matrix(),
    atol=1e-10,
)
assert_close(
    "Recovered 9x9 matches original solved-state covariance",
    solved_roundtrip_9x9.to_matrix()[0],
    keplerian_covariance[0],
    atol=1e-9,
)


## Synthetic Case 2: `7x7` Collapse by `object_id`

Here we build three variants for one object with a solved-state covariance that includes `A2`.

The goal is to verify that `collapse_by_object_id()` now does the right thing for the full solved-state uncertainty, not just the orbital `6x6`.


In [ ]:
base_7x7 = np.diag([1e-8, 2e-8, 3e-8, 4e-10, 5e-10, 6e-10, 9e-26])
variant_orbits = VariantOrbits.from_kwargs(
    orbit_id=["obj1", "obj1", "obj1"],
    object_id=["obj1", "obj1", "obj1"],
    variant_id=["0", "1", "2"],
    weights=[1 / 3, 1 / 3, 1 / 3],
    weights_cov=[1 / 3, 1 / 3, 1 / 3],
    physical_parameters=PhysicalParameters.from_kwargs(H_v=[15.0, 15.0, 15.0], G=[0.15, 0.15, 0.15]),
    non_gravitational_parameters=NonGravitationalParameters.from_kwargs(
        source=["SBDB"] * 3,
        model=["nongrav"] * 3,
        solution_dimension=[7] * 3,
        parameter_count=[1] * 3,
        estimated_parameter_names=["A2"] * 3,
        A1=[None] * 3,
        A1_sigma=[None] * 3,
        A2=[1e-13, 1.2e-13, 0.8e-13],
        A2_sigma=[None] * 3,
        A3=[None] * 3,
        A3_sigma=[None] * 3,
        DT=[None] * 3,
        DT_sigma=[None] * 3,
        R0=[None] * 3,
        R0_sigma=[None] * 3,
        ALN=[None] * 3,
        ALN_sigma=[None] * 3,
        NK=[None] * 3,
        NK_sigma=[None] * 3,
        NM=[None] * 3,
        NM_sigma=[None] * 3,
        NN=[None] * 3,
        NN_sigma=[None] * 3,
        AMRAT=[None] * 3,
        AMRAT_sigma=[None] * 3,
        RHO=[None] * 3,
        RHO_sigma=[None] * 3,
    ),
    solved_state_covariance=SolvedStateCovariances.from_matrix(
        [base_7x7, base_7x7, base_7x7],
        [["x", "y", "z", "vx", "vy", "vz", "A2"]] * 3,
    ),
    coordinates=CartesianCoordinates.from_kwargs(
        x=[1.0, 1.1, 0.9],
        y=[1.0, 1.1, 0.9],
        z=[1.0, 1.1, 0.9],
        vx=[0.1, 0.11, 0.09],
        vy=[0.1, 0.11, 0.09],
        vz=[0.1, 0.11, 0.09],
        time=Timestamp.from_mjd([60000.0, 60000.0, 60000.0]),
        origin=Origin.from_kwargs(code=["SUN", "SUN", "SUN"]),
        frame="ecliptic",
    ),
)

collapsed = variant_orbits.collapse_by_object_id()
display(nongrav_summary(collapsed))
display(orbital_covariance_frame("collapsed obj1", collapsed.coordinates, collapsed.solved_state_covariance))

assert_close(
    "Collapsed recovered 6x6 matches collapsed coordinate covariance",
    collapsed.solved_state_covariance.to_orbital_covariances().to_matrix(),
    collapsed.coordinates.covariance.to_matrix(),
    atol=1e-12,
)


## Real Case 1: SBDB `99942` (`8x8`)

This fixture is a good real-world check because the fitted solved-state covariance includes `A1` and `A2`, giving us an `8x8` matrix after conversion into the internal Cartesian solved-state basis.

We verify three things:

- the real fixture is ingested with the expected solved-state dimension and parameter ordering
- the recovered orbital `6x6` matches the coordinate covariance as ingested
- an orbit-level conversion with `include_nongrav=True` preserves the solved-state covariance through a Cartesian -> Keplerian -> Cartesian round-trip


In [ ]:
orbit_99942 = load_sbdb_orbit("99942_phys.json", object_id="99942")
display(nongrav_summary(orbit_99942))
display(orbital_covariance_frame("SBDB 99942 input", orbit_99942.coordinates, orbit_99942.solved_state_covariance))

keplerian_99942, solved_keplerian_99942 = orbit_99942.to_keplerian(include_nongrav=True)
cartesian_99942_roundtrip, solved_cartesian_99942_roundtrip = transform_coordinates(
    keplerian_99942,
    representation_out=CartesianCoordinates,
    solved_state_covariances=solved_keplerian_99942,
)

display(compare_orbital_blocks(
    "SBDB 99942 original 6x6 vs recovered roundtrip 6x6",
    orbit_99942.coordinates.covariance.to_matrix()[0],
    solved_cartesian_99942_roundtrip.to_orbital_covariances().to_matrix()[0],
))

assert_close(
    "SBDB 99942 recovered 6x6 matches input 6x6",
    solved_cartesian_99942_roundtrip.to_orbital_covariances().to_matrix(),
    orbit_99942.coordinates.covariance.to_matrix(),
    atol=1e-10,
)
assert_close(
    "SBDB 99942 roundtrip coordinate covariance stays aligned",
    solved_cartesian_99942_roundtrip.to_orbital_covariances().to_matrix(),
    cartesian_99942_roundtrip.covariance.to_matrix(),
    atol=1e-10,
)


## Real Case 2: NEOCC `99942` (`7x7`)

This fixture exercises the NEOCC path, which carries a `7x7` solved-state covariance with `A2`.

We use the same checks as above, but now against the NEOCC parser and solved-state transformation path.


In [ ]:
orbit_99942_neocc = load_neocc_orbit("99942.ke1")
display(nongrav_summary(orbit_99942_neocc))
display(orbital_covariance_frame("NEOCC 99942 input", orbit_99942_neocc.coordinates, orbit_99942_neocc.solved_state_covariance))

cometary_99942_neocc, solved_cometary_99942_neocc = orbit_99942_neocc.to_cometary(include_nongrav=True)
cartesian_99942_neocc_roundtrip, solved_cartesian_99942_neocc_roundtrip = transform_coordinates(
    cometary_99942_neocc,
    representation_out=CartesianCoordinates,
    solved_state_covariances=solved_cometary_99942_neocc,
)

display(compare_orbital_blocks(
    "NEOCC 99942 original 6x6 vs recovered roundtrip 6x6",
    orbit_99942_neocc.coordinates.covariance.to_matrix()[0],
    solved_cartesian_99942_neocc_roundtrip.to_orbital_covariances().to_matrix()[0],
))

assert_close(
    "NEOCC 99942 recovered 6x6 matches input 6x6",
    solved_cartesian_99942_neocc_roundtrip.to_orbital_covariances().to_matrix(),
    orbit_99942_neocc.coordinates.covariance.to_matrix(),
    atol=1e-8,
)
assert_close(
    "NEOCC 99942 roundtrip coordinate covariance stays aligned",
    solved_cartesian_99942_neocc_roundtrip.to_orbital_covariances().to_matrix(),
    cartesian_99942_neocc_roundtrip.covariance.to_matrix(),
    atol=1e-8,
)


## Variant Creation With and Without Non-Grav Sampling

This is the practical downstream check.

For the synthetic `9x9` orbit, we create variants twice:

- once with `include_nongrav=True`, which should sample `A1/A2/A3` as part of the joint solved state
- once with `include_nongrav=False`, which should fall back to the orbital `6x6` only and strip the nongrav state from the variants


In [ ]:
joint_orbit = Orbits.from_kwargs(
    orbit_id=["joint"],
    object_id=["joint"],
    physical_parameters=PhysicalParameters.from_kwargs(H_v=[20.0], G=[0.15]),
    non_gravitational_parameters=NonGravitationalParameters.from_kwargs(
        source=["SBDB"],
        model=["nongrav"],
        solution_dimension=[9],
        parameter_count=[3],
        estimated_parameter_names=["A1,A2,A3"],
        A1=[1.0e-13],
        A1_sigma=[3.0e-13],
        A2=[-2.0e-13],
        A2_sigma=[2.0e-13],
        A3=[4.0e-13],
        A3_sigma=[1.0e-13],
        DT=[None], DT_sigma=[None],
        R0=[None], R0_sigma=[None],
        ALN=[None], ALN_sigma=[None],
        NK=[None], NK_sigma=[None],
        NM=[None], NM_sigma=[None],
        NN=[None], NN_sigma=[None],
        AMRAT=[None], AMRAT_sigma=[None],
        RHO=[None], RHO_sigma=[None],
    ),
    solved_state_covariance=SolvedStateCovariances.from_matrix(
        keplerian_covariance,
        [["x", "y", "z", "vx", "vy", "vz", "A1", "A2", "A3"]],
    ),
    coordinates=CartesianCoordinates.from_kwargs(
        x=[1.0], y=[2.0], z=[3.0],
        vx=[0.01], vy=[0.02], vz=[0.03],
        covariance=CoordinateCovariances.from_matrix(keplerian_covariance[:, :6, :6]),
        time=Timestamp.from_mjd([60000.0]),
        origin=Origin.from_kwargs(code=["SUN"]),
        frame="ecliptic",
    ),
)

variants_joint = VariantOrbits.create(joint_orbit, method="sigma-point", include_nongrav=True)
variants_orbital_only = VariantOrbits.create(joint_orbit, method="sigma-point", include_nongrav=False)

variant_summary = pd.DataFrame({
    "case": ["joint solved-state sampling", "orbital-only sampling"],
    "variant_count": [len(variants_joint), len(variants_orbital_only)],
    "unique_A1_values": [len(set(variants_joint.non_gravitational_parameters.A1.to_pylist())), len(set(variants_orbital_only.non_gravitational_parameters.A1.to_pylist()))],
    "unique_A2_values": [len(set(variants_joint.non_gravitational_parameters.A2.to_pylist())), len(set(variants_orbital_only.non_gravitational_parameters.A2.to_pylist()))],
    "solved_state_dimension": [
        variants_joint.solved_state_covariance.dimension[0].as_py(),
        variants_orbital_only.solved_state_covariance.dimension[0].as_py(),
    ],
})
display(variant_summary)


## Takeaways

If all of the checks above pass, we have evidence that:

- the full solved-state covariance survives supported coordinate conversions
- the orbital `6x6` recovered from the `6 + k` matrix stays aligned with the ordinary coordinate covariance
- `collapse_by_object_id()` rebuilds the solved-state covariance instead of dropping back to orbital-only behavior
- `VariantOrbits.create()` can either include or exclude nongrav sampling explicitly

This notebook is intended as a verification companion to the earlier non-grav propagation notebook.
